## 5. Stationarity Tests

ADF and KPSS are diagnostics, not an automatic modelling oracle. The conservative rule used here is ADF p-value `< 0.05` together with KPSS p-value `> 0.05`, but economic interpretation still matters. Index/price-level variables can often be represented by log differences, while rate variables such as the cash rate and unemployment rate are usually more meaningful as percentage-point changes. Any `I(2)` result is flagged for caution rather than treated as unquestioned evidence that second differencing must be used.

In [7]:
eda_df = df.set_index('quarter_period')
STATIONARITY_COLS = [column for column in CPI_FAMILY_BASE_COLS + CPI_DERIVED_DIAGNOSTIC_COLS if column in df_model.columns] + EXTERNAL_BASE_COLS

def clean_series(column: str) -> pd.Series:
    return pd.to_numeric(eda_df[column], errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()


def safe_adf(series: pd.Series) -> float:
    series = series.dropna()
    if len(series) < 12 or series.nunique() < 3:
        return np.nan
    try:
        return float(adfuller(series, autolag='AIC')[1])
    except Exception:
        return np.nan


def safe_kpss(series: pd.Series) -> float:
    series = series.dropna()
    if len(series) < 12 or series.nunique() < 3:
        return np.nan
    try:
        return float(kpss(series, regression='c', nlags='auto')[1])
    except Exception:
        return np.nan


def is_growth_or_change_column(column: str) -> bool:
    return column in set(CPI_FAMILY_BASE_COLS + CPI_DERIVED_DIAGNOSTIC_COLS) or column.endswith('_growth') or column.endswith('_change')


RATE_LEVEL_COLUMNS = {'cash_rate', 'unemployment_rate', 'inflation_expectations_business'}

def should_use_log_diff(column: str, series: pd.Series) -> bool:
    return column not in RATE_LEVEL_COLUMNS and not is_growth_or_change_column(column) and len(series) > 0 and float((series > 0).mean()) >= 0.80


def stationarity_variants(column: str, series: pd.Series) -> dict[str, pd.Series]:
    variants = {
        'level': series,
        'diff1': series.diff(),
        'diff2': series.diff().diff(),
    }
    variants['log_diff1'] = np.log(series.where(series > 0)).diff() if should_use_log_diff(column, series) else pd.Series(dtype='float64')
    return variants


def integration_order_from_tests(row: pd.Series) -> str:
    def stationary(prefix: str) -> bool:
        adf_ok = pd.notna(row[f'{prefix}_adf_p']) and row[f'{prefix}_adf_p'] < 0.05
        kpss_ok = pd.notna(row[f'{prefix}_kpss_p']) and row[f'{prefix}_kpss_p'] > 0.05
        return bool(adf_ok and kpss_ok)
    if stationary('level'):
        return 'I(0)'
    if stationary('diff1') or stationary('log_diff1'):
        return 'I(1)'
    if stationary('diff2'):
        return 'I(2)'
    return 'unclear'


stationarity_rows = []
for column in STATIONARITY_COLS:
    series = clean_series(column)
    row = {'variable': column, 'n_obs': int(len(series))}
    for variant_name, variant_series in stationarity_variants(column, series).items():
        row[f'{variant_name}_adf_p'] = safe_adf(variant_series)
        row[f'{variant_name}_kpss_p'] = safe_kpss(variant_series)
    stationarity_rows.append(row)

stationarity = pd.DataFrame(stationarity_rows)
stationarity['integration_order'] = stationarity.apply(integration_order_from_tests, axis=1)
stationarity['interpretation_note'] = np.select(
    [
        stationarity['variable'].isin(RATE_LEVEL_COLUMNS),
        stationarity['integration_order'].eq('I(2)'),
        stationarity['variable'].str.endswith('_change'),
    ],
    [
        'rate level: prefer percentage-point changes for Elastic Net screening',
        'I(2) diagnostic: verify cautiously before using second differences',
        'rate/growth change feature: already an economically interpretable transform',
    ],
    default='',
)
display(stationarity.round(4))

,variable,n_obs,level_adf_p,level_kpss_p,diff1_adf_p,diff1_kpss_p,diff2_adf_p,diff2_kpss_p,log_diff1_adf_p,log_diff1_kpss_p,integration_order,interpretation_note
0,cpi_qoq,124,0.0000,0.100,0.0000,0.1000,0.0000,0.1000,NaN,NaN,I(0),
1,cpi_yoy,124,0.0299,0.100,0.0000,0.1000,0.0000,0.1000,NaN,NaN,I(0),
2,trimmed_mean_cpi_qoq,124,0.0237,0.100,0.0000,0.1000,0.0000,0.1000,NaN,NaN,I(0),
3,trimmed_mean_cpi_yoy,124,0.1896,0.100,0.0000,0.1000,0.0000,0.1000,NaN,NaN,I(1),
4,headline_trimmed_mean_yoy_gap,124,0.0001,0.100,0.0000,0.1000,0.0000,0.1000,NaN,NaN,I(0),
5,unemployment_rate,124,0.2713,0.010,0.0000,0.1000,0.0000,0.1000,NaN,NaN,I(1),rate level: prefer percentage-point changes fo...
6,unemployment_rate_change,123,0.0000,0.100,0.0000,0.1000,0.0000,0.1000,NaN,NaN,I(0),rate/growth change feature: already an economi...
7,cash_rate,124,0.1866,0.010,0.0000,0.1000,0.0000,0.1000,NaN,NaN,I(1),rate level: prefer percentage-point changes fo...
8,cash_rate_change,123,0.0000,0.100,0.0000,0.1000,0.0000,0.0417,NaN,NaN,I(0),rate/growth change feature: already an economi...
9,wage_price_index,114,0.9948,0.010,0.3847,0.0187,0.0031,0.1000,0.3866,0.0209,I(2),I(2) diagnostic: verify cautiously before usin...
